In [0]:
# ==============================================================================
# END-TO-END DEMO: AUTO LOADER INGESTION & COMPLETE SCHEMA INFERENCE VERIFICATION
# ==============================================================================

# --- STEP 1: ENVIRONMENT & STORAGE SETUP ---
spark.sql("CREATE CATALOG IF NOT EXISTS training_medallion")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.00_landing")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.01_bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS training_medallion.00_landing.raw_landing_zone")

base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"

# Create landing directories
dbutils.fs.mkdirs(f"{base_volume}/json_landing")
dbutils.fs.mkdirs(f"{base_volume}/csv_landing")
print(f"Directory ready at: {base_volume}")


# --- STEP 2: SIMULATE INCOMING RAW DATA ---
# Raw JSON Batch
json_payload = """{"ticket_id": "TCK-101", "customer_id": 1, "issue": "Payment failed", "status": "OPEN"}
{"ticket_id": "TCK-102", "customer_id": 2, "issue": "Account locked", "status": "CLOSED"}"""
dbutils.fs.put(f"{base_volume}/json_landing/tickets_batch1.json", json_payload, overwrite=True)

# Raw CSV Batch
csv_payload = """customer_id,customer_name,region,total_orders\n1,Jenna,APAC,2\n2,John,EMEA,6\n3,Lea,AMER,10"""
dbutils.fs.put(f"{base_volume}/csv_landing/customers_batch1.csv", csv_payload, overwrite=True)
print("Raw JSON and CSV files successfully staged.")


# --- STEP 3: CONFIGURE AUTO LOADER READ STREAMS ---
checkpoint_json = f"{base_volume}/_checkpoints/json_tickets"
schema_json = f"{base_volume}/_schemas/json_tickets"

df_json_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_json)
    .load(f"{base_volume}/json_landing")
)

checkpoint_csv = f"{base_volume}/_checkpoints/csv_customers"
schema_csv = f"{base_volume}/_schemas/csv_customers"

df_csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_csv)
    .load(f"{base_volume}/csv_landing")
)


# --- STEP 4: PROOF 1 - STREAMING IN-MEMORY SCHEMA INFERENCE ---
# Displays schema inferred purely from raw files before Delta writing
print("\n[PROOF 1] Streaming In-Memory Inferred Schema (JSON):")
df_json_stream.printSchema()

print("\n[PROOF 1] Streaming In-Memory Inferred Schema (CSV):")
df_csv_stream.printSchema()


# --- STEP 5: WRITE MICRO-BATCH TO DELTA BRONZE TABLES ---
q_json = (
    df_json_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_json)
    .table("training_medallion.01_bronze.bronze_json_tickets")
)
q_json.awaitTermination()

q_csv = (
    df_csv_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_csv)
    .table("training_medallion.01_bronze.bronze_csv_customers")
)
q_csv.awaitTermination()
print("Micro-batch ingestion completed.")


# --- STEP 6: PROOF 2 - SCHEMA PERSISTENCE FILES IN VOLUME ---
# Check the base schema directory directly rather than hardcoding /_schema
try:
    print(f"\n[PROOF 2] Metadata Directory Content ({schema_json}):")
    schema_dir_contents = dbutils.fs.ls(schema_json)
    display(schema_dir_contents)
    
    # Recursively locate and display the actual inferred schema file
    for item in schema_dir_contents:
        if item.isDir():
            print(f"\nInspecting subfolder: {item.path}")
            display(dbutils.fs.ls(item.path))
        else:
            print(f"\nFound schema file: {item.path}")
            print(dbutils.fs.head(item.path, 500))
except Exception as e:
    print(f"Directory inspection notice: {e}")


# --- STEP 7: PROOF 3 - FINAL DELTA TABLE SCHEMA & DATA VALIDATION ---
print("\n[PROOF 3] JSON Target Table Schema Definition:")
display(spark.sql("DESCRIBE TABLE EXTENDED training_medallion.01_bronze.bronze_json_tickets"))

print("\n[PROOF 3] Data Type Verification via typeof() Expressions:")
display(
    spark.table("training_medallion.01_bronze.bronze_json_tickets")
    .selectExpr(
        "ticket_id",
        "typeof(ticket_id) AS type_ticket_id",
        "customer_id",
        "typeof(customer_id) AS type_customer_id",
        "status"
    )
)

Directory ready at: /Volumes/training_medallion/00_landing/raw_landing_zone
Wrote 177 bytes.
Wrote 88 bytes.
Raw JSON and CSV files successfully staged.

[PROOF 1] Streaming In-Memory Inferred Schema (JSON):
root
 |-- customer_id: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- status: string (nullable = true)
 |-- ticket_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)


[PROOF 1] Streaming In-Memory Inferred Schema (CSV):
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- total_orders: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

Micro-batch ingestion completed.

[PROOF 2] Metadata Directory Content (/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets):


path,name,size,modificationTime
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets/_schemas/,_schemas/,0,1793284697854



Inspecting subfolder: dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets/_schemas/


path,name,size,modificationTime
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets/_schemas/0,0,427,1792940110000



[PROOF 3] JSON Target Table Schema Definition:


col_name,data_type,comment
customer_id,string,null
issue,string,null
status,string,null
ticket_id,string,null
_rescued_data,string,null
,,
# Delta Statistics Columns,,
Column Names,"ticket_id, customer_id, issue, status, _rescued_data",
Column Selection Method,first-32,
,,



[PROOF 3] Data Type Verification via typeof() Expressions:


ticket_id,type_ticket_id,customer_id,type_customer_id,status
TCK-104,string,4,string,OPEN
TCK-105,string,5,string,CLOSED
TCK-101,string,1,string,OPEN
TCK-102,string,2,string,CLOSED
